In [18]:
from ast import *
from utils import *
import import_ipynb
from x86_ast import *
from rco_test import *
from select_instr import *

In [19]:
 def assign_homes_arg(a: arg, home: Dict[Variable, arg]) -> arg:
        # YOUR CODE HERE
        match a:
            case Variable(val):               
                return Deref('rbp',home[Variable(val)])
            case Immediate(val):
                return Immediate(val)
            case Reg(reg):
                return Reg(reg)
            case _:
                return arg
        

In [20]:
 def assign_homes_instr(i: instr,
                           home: Dict[Variable, arg]) -> instr:
        # YOUR CODE HERE
        
        match i:
            case Instr('movq',[Immediate(int),Variable(id)]):
                if Variable(id) not in home:
                    home[Variable(id)] = (len(home) + 1) * -8,
                    return Instr('movq',[assign_homes_arg(Immediate(int),home), assign_homes_arg(Variable(id),home)])
                else:
                    return Instr('movq',[assign_homes_arg(Immediate(int),home), assign_homes_arg(Variable(id), home)])
            case Instr('movq', [Variable(a), Variable(b)]):
                if Variable(b) not in home:
                    home[Variable(b)] = (len(home) + 1) * -8
                    return Instr('movq',[assign_homes_arg(Variable(a), home), assign_homes_arg(Variable(b),home)])
                else:
                    return Instr('movq', [assign_homes_arg(Variable(a),home) , assign_homes_arg(Variable(b),home)])
            case Instr('movq', [Variable(a), Reg(reg)]):
                if Variable(a) not in home:
                     home[Variable(a)] = (len(home) + 1) * -8
                return Instr('movq', [assign_homes_arg(Variable(a),home), assign_homes_arg(Reg(reg),home)] )
            case Instr('movq',[Reg(reg), Variable(a)]):
                  if Variable(a) not in home:
                       home[Variable(a)] = (len(home) + 1) * -8
                  return Instr('movq',[assign_homes_arg(Reg(reg),home), assign_homes_arg(Variable(a),home)])
            case Instr('addq',[Immediate(int), Variable(id)]):
                if Variable(id) not in home:
                    home[Variable(id)] = (len(home) + 1) * -8
                return Instr('addq',[assign_homes_arg(Immediate(int),home), assign_homes_arg(Variable(id),home)])
            case Instr('subq',[Immediate(int), Variable(id)]):
                if Variable(id) not in home:
                    home[Variable(id)] = (len(home) + 1) * -8
                return Instr('subq',[assign_homes_arg(Immediate(int),home), assign_homes_arg(Variable(id),home)])
            case Instr('addq',[Variable(id), Reg(reg)]):
                if Variable(id) not in home:
                    home[Variable(id)] = (len(home)+1) * -8
                return Instr('addq',[assign_homes_arg(Variable(id),home), assign_homes_arg(Reg(reg),home)])
            case Instr('subq',[Variable(id),Reg(reg)]):
                if Variable(id) not in home:
                    home[Variable(id)] = (len(home) + 1) * -8
                return Instr('subq',[assign_homes_arg(Variable(id),home), assign_homes_arg(Reg(reg),home)])
            
            case _:
                return i
                
                return
                


In [21]:
 def assign_homes(p: X86Program) -> X86Program:
        # YOUR CODE HERE
    
        new_list = []
        home = {}
        for instr in p.body:          
                    new_list.append(assign_homes_instr(instr,home))
        return X86Program(new_list)

In [22]:
if __name__ == "__main__":
    import textwrap
    code = textwrap.dedent("""
    a = (42 + 13) + -7
    b = a + 6
    print(b + 1)""")
    parsed_code = parse(code)
    rco_code = remove_complex_operands(parsed_code)
    select_instr_code = select_instruction(rco_code)
    print(select_instr_code)
    assign_homes_code = assign_homes(select_instr_code)
    print(assign_homes_code)

	.globl main
main:
    movq $42, %rax
    addq $13, %rax
    movq %rax, temp.10
    movq $7, %rax
    negq %rax
    movq %rax, temp.11
    movq temp.10, %rax
    addq temp.11, %rax
    movq %rax, a
    movq a, %rax
    addq $6, %rax
    movq %rax, b
    movq b, %rax
    addq $1, %rax
    movq %rax, temp.12
    movq temp.12, %rdi
    callq print_int


	.globl main
main:
    movq $42, %rax
    addq $13, %rax
    movq %rax, -8(%rbp)
    movq $7, %rax
    negq %rax
    movq %rax, -16(%rbp)
    movq -8(%rbp), %rax
    addq -16(%rbp), %rax
    movq %rax, -24(%rbp)
    movq -24(%rbp), %rax
    addq $6, %rax
    movq %rax, -32(%rbp)
    movq -32(%rbp), %rax
    addq $1, %rax
    movq %rax, -40(%rbp)
    movq -40(%rbp), %rdi
    callq print_int


